## 0. Hugging Face Authentication (Optional)
Setting a token avoids rate limits and speeds up model downloads.

In [ ]:
import os
# Replace with your actual token from https://huggingface.co/settings/tokens
# os.environ["HF_TOKEN"] = "your_token_here"
print("HF Token set!" if "HF_TOKEN" in os.environ else "HF Token not set (using unauthenticated requests)")

# ADR Security Validator - AMD Developer Cloud Deployment Guide

This notebook guides you through the deployment of the ADR Security Validator on an AMD Instinct™ GPU (like MI300X or MI210) using the ROCm™ software stack.

### ⚠️ Troubleshooting: "getcwd: cannot access parent directories"
If you see this error, it means the directory you were in was deleted. Run the cell below to fix the working directory.

In [ ]:
import os
project_dir = "/root/AMD-Developer-Hackathon"
if os.path.exists(project_dir):
    %cd {project_dir}
print(f"Working directory set to: {os.getcwd()}")

## 1. Verify GPU Status

In [ ]:
!rocm-smi

## 2. Environment Setup
Install the optimized version of PyTorch for ROCm and other project dependencies.

In [ ]:
# Install PyTorch optimized for ROCm 6.2
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/rocm6.2

# Install project requirements
!pip install -r backend/requirements.txt

# Install optimized training library Unsloth for ROCm
!pip install "unsloth[rocm] @ git+https://github.com/unslothai/unsloth.git"

## 3. Vector Database (Qdrant)
Start the Qdrant vector database using Docker Compose. This ensures persistence and better service management.

In [ ]:
!docker compose up -d
!docker compose ps

## 4. Data Indexing
Index the ADR data from Kubernetes, Django, and Rust into Qdrant.

In [ ]:
%cd backend
!python qdrant_setup.py --embed
%cd ..

## 4.5 Generate Training Dataset
Create the fine-tuning dataset from ADRs. This must run before fine-tuning.

In [ ]:
%cd fine-tuning
!python generate_dataset.py
%cd ..

## 5. Fine-tuning (Optional)
Run the fine-tuning script to optimize Qwen3-8B for ADR validation and security analysis. On an MI300X, this takes about 1.5 hours.

In [ ]:
# The fine_tune_qwen3_8b.py script includes ROCm + Python 3.12 fixes
# (torch.int1/int2 patches and transformers group_mm_fallback workaround)
%cd fine-tuning
!python fine_tune_qwen3_8b.py
%cd ..

## 6. Start the API Backend
Launch the FastAPI server. 

**Note:** To keep the server running, it's recommended to run this in a terminal or use a tool like `nohup` or `screen`. 

```bash
cd backend
uvicorn main:app --host 0.0.0.0 --port 8000
```

## 7. Verification & Debugging
This section helps you verify that the entire system (Vector DB + Fine-tuned Model) is working correctly.

### 7.1 Verify Vector Database (Qdrant)
Check if the ADRs were correctly indexed and are searchable.

In [ ]:
import requests
try:
    response = requests.get("http://localhost:6333/collections/adrs")
    data = response.json()
    print(f"Collection Status: {data['status']}")
    print(f"Points Indexed: {data['result']['points_count']}")
except Exception as e:
    print(f"Error connecting to Qdrant: {e}")

### 7.2 Full System Test (The Warehouse Management System Example)
This ADR is designed to trigger security rules, find contradictions in the Vector DB, and elicit a critique from the fine-tuned AI model.

In [ ]:
import requests
import json

url = "http://localhost:8000/validate-adr"
payload = {
    "title": "Arquitectura para el Sistema Central de Gestión de Almacenes (WMS)",
    "content": """
## Contexto
Necesitamos modernizar el sistema de inventario para nuestros 15 almacenes regionales. El sistema debe manejar miles de movimientos de stock por hora.

## Decision
1. Frontend: React con Vite.
2. Backend: Microservicios en Python (FastAPI) en contenedores Docker.
3. Base de Datos: Una unica instancia de PostgreSQL centralizada.
4. Infraestructura: Docker Compose en servidores locales.
5. Seguridad: Credenciales guardadas directamente en el archivo config.py del backend y uso del puerto 5432 sin SSL para agilizar la red local.
    """,
    "context": "Proyecto de modernizacion de logistica 2026"
}

try:
    response = requests.post(url, json=payload)
    result = response.json()
    print(json.dumps(result, indent=2, ensure_ascii=False))
except Exception as e:
    print(f"Error: {e}. Make sure the API server is running on port 8000.")

### 7.3 How to know if it worked?
- **Vector DB Working?**: Look at `related_adrs`. If you see a list of actual ADRs with similarity scores, semantic search is active.
- **Fine-tuning working?**: Look at `recommendations`. If the first element starts with `AI Analysis:`, that is the output of your fine-tuned Qwen3 model.
- **Security Rules Working?**: Look at `security_risks`. It should detect the `hardcoded_credentials` risk from the `config.py` mention.